# Hotel Bookings: 0NF → 3NF, executed against PostgreSQL

The source is one wide spreadsheet of 49 hotel bookings in **unnormalised form (0NF)**:
every descriptive attribute repeated inline as text.

This notebook takes it to **third normal form**, creating the schema in PostgreSQL through
SQLAlchemy and loading the data, then proving the decomposition lost nothing. Each step
states which normal form it satisfies and shows the evidence for it, rather than asserting
it.

| Step | Normal form | What it fixes |
|---|---|---|
| 1 | 1NF | one atomic, consistently-typed value per field |
| 2 | 2NF | no non-key attribute depending on part of a key |
| 3 | 3NF | no non-key attribute depending on another non-key attribute |
| 4 | — | create the schema in PostgreSQL (constraints enforce it) |
| 5 | — | load, then verify the decomposition is lossless |

In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, inspect, text

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Credentials come from the environment - see .env.example at the repository root.
USER = os.environ.get("POSTGRES_USER", "postgres")
PWD  = os.environ.get("POSTGRES_PASSWORD")
HOST = os.environ.get("POSTGRES_HOST", "localhost")
PORT = os.environ.get("POSTGRES_PORT", "5432")
DB   = os.environ.get("HOTEL_DB", "hotel")

engine = create_engine(f"postgresql+psycopg2://{USER}:{PWD}@{HOST}:{PORT}/{DB}")
with engine.connect() as con:
    print(con.execute(text("SELECT version()")).scalar().split(",")[0])

PostgreSQL 17.12 on x86_64-windows


## Step 0 — the 0NF source

One table, 15 columns, no keys declared and no constraints. Every booking carries its own
copy of the meal-plan name, room-type name, market source and status as free text.

In [2]:
ROOT = Path.cwd().parent
raw = pd.read_excel(ROOT / "data" / "hotel_0nf.xlsx")
print(f"{raw.shape[0]} rows x {raw.shape[1]} columns")
raw.head()

49 rows x 15 columns


,Booking_ID,Guest_ID,adults,children,weekend nights,week nights,meal,parking,room type,lead time,source,price,requests,date of reservation,booking status
0,INN00001,GI00001,1,1,2,5,Meal Plan 1,0,Room_Type 1,224,Offline,88.00,0,2015-02-10 00:00:00,Not_Canceled
1,INN00002,GI00002,1,0,1,3,Meal Plan 1,0,Room_Type 1,5,Online,106.68,1,2018-06-11 00:00:00,Not_Canceled
2,INN00003,GI00003,2,1,1,3,Meal Plan 1,0,Room_Type 2,1,Online,50.00,0,2/28/2018,Canceled
3,INN00004,GI00004,1,0,0,2,Meal Plan 1,0,Room_Type 2,211,Online,100.00,1,5/20/2017,Canceled
4,INN00005,GI00005,1,0,1,2,Meal Plan 2,0,Room_Type 1,48,Online,77.00,0,2018-11-04 00:00:00,Canceled


## Step 1 — First normal form

1NF requires each field to hold a single atomic value of one consistent type.

The violation here is not obvious from looking at the table: `date of reservation` holds
**two different representations in the same column**.

In [3]:
dates = raw["date of reservation"].astype(str)
iso = dates.str.match(r"^\d{4}-\d{2}-\d{2}")
us  = dates.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")

print(f"ISO  'yyyy-mm-dd hh:mm:ss' : {iso.sum()} rows   e.g. {dates[iso].iloc[0]}")
print(f"US   'm/d/yyyy'            : {us.sum()} rows   e.g. {dates[us].iloc[0]}")
print(f"neither                    : {len(dates) - iso.sum() - us.sum()} rows")

ISO  'yyyy-mm-dd hh:mm:ss' : 16 rows   e.g. 2015-02-10 00:00:00
US   'm/d/yyyy'            : 33 rows   e.g. 2/28/2018
neither                    : 0 rows


Why this matters more than it looks: parsing with a single format silently converts the
other two thirds to `NaT`. Nothing raises, and the warehouse quietly loses 33 of 49 dates.

MySQL, which the original coursework used, accepted the mixed strings on insert into a
`DATE` column. **PostgreSQL rejects them**, which is how the problem came to light at all —
the stricter database surfaced a latent data-quality bug.

In [4]:
def parse_reservation_date(value):
    """Try each known format; fail loudly rather than returning NaT."""
    s = str(value).strip()
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y"):
        try:
            return pd.to_datetime(s, format=fmt).date()
        except (ValueError, TypeError):
            continue
    raise ValueError(f"unparseable reservation date: {value!r}")


# naive single-format parsing, for comparison
naive = pd.to_datetime(raw["date of reservation"], format="%Y-%m-%d %H:%M:%S",
                       errors="coerce")
print(f"single format  -> {naive.isna().sum()} of {len(raw)} dates lost as NaT")

parsed = raw["date of reservation"].map(parse_reservation_date)
print(f"format-aware   -> {parsed.isna().sum()} lost")
print(f"range: {min(parsed)} to {max(parsed)}")

single format  -> 33 of 49 dates lost as NaT
format-aware   -> 0 lost
range: 2015-02-10 to 2018-12-26


In [5]:
RENAME = {
    "Booking_ID": "booking_id", "Guest_ID": "guest_id",
    "weekend nights": "weekend_nights", "week nights": "week_nights",
    "lead time": "lead_time", "requests": "special_requests",
    "parking": "car_parking", "date of reservation": "reservation_date",
}

one_nf = raw.rename(columns=RENAME).copy()
one_nf["reservation_date"] = parsed
one_nf["car_parking"] = one_nf["car_parking"].astype(int).astype(bool)   # was '0'/'1' text
for c in ("adults", "children", "weekend_nights", "week_nights",
          "lead_time", "special_requests"):
    one_nf[c] = one_nf[c].astype(int)
one_nf["price"] = one_nf["price"].astype(float).round(2)
for c in ("meal", "room type", "source", "booking status"):
    one_nf[c] = one_nf[c].astype(str).str.strip()

one_nf.dtypes.to_frame("dtype").T

,booking_id,guest_id,adults,children,weekend_nights,week_nights,meal,car_parking,room type,lead_time,source,price,special_requests,reservation_date,booking status
dtype,object,object,int64,int64,int64,int64,object,bool,object,int64,object,float64,int64,object,object


Column names with spaces are also renamed to `snake_case`. They required backtick quoting
in MySQL and would need double quotes in PostgreSQL in every query thereafter — a cost paid
forever to save a rename once.

## Step 2 — Second normal form

2NF concerns **partial dependencies**: a non-key attribute depending on only part of a
composite key. So the first question is whether a composite key exists at all.

The original coursework split guest attributes into their own table and described that as
removing a partial dependency. Let me check whether that holds.

In [6]:
n = len(one_nf)
print(f"rows                  : {n}")
print(f"distinct booking_id   : {one_nf.booking_id.nunique()}")
print(f"distinct guest_id     : {one_nf.guest_id.nunique()}")
print(f"booking_id unique?    : {one_nf.booking_id.is_unique}")
print(f"guest_id unique?      : {one_nf.guest_id.is_unique}")

per_guest = one_nf.groupby("guest_id").size()
print(f"\nbookings per guest    : min {per_guest.min()}, max {per_guest.max()}")

rows                  : 49
distinct booking_id   : 49
distinct guest_id     : 49
booking_id unique?    : True
guest_id unique?      : True

bookings per guest    : min 1, max 1


**Both are unique across all 49 rows — a strict 1:1.** `booking_id` alone identifies a
row, so there is no composite key, and therefore no partial dependency for 2NF to remove.
The table is already in 2NF once it is in 1NF.

The original's "2NF step" removed something that was not there. A `guest` table is still
created below, but for a different and honest reason: it is the correct shape for the
domain, since a hotel guest can obviously book twice. On *this* 49-row extract it is a
**degenerate dimension** — it removes no redundancy. That is a different claim from saying
it normalises something.

## Step 3 — Third normal form

3NF concerns **transitive dependencies**: a non-key attribute determined by another non-key
attribute rather than by the key. The candidates are attributes whose value repeats across
many rows because they describe a *category*, not a booking.

In [7]:
candidates = ["meal", "room type", "source", "booking status",
              "car_parking", "lead_time", "price"]
summary = pd.DataFrame({
    "distinct": [one_nf[c].nunique() for c in candidates],
    "rows_per_value": [round(n / one_nf[c].nunique(), 1) for c in candidates],
    "dtype": [str(one_nf[c].dtype) for c in candidates],
}, index=candidates).sort_values("distinct")
summary

,distinct,rows_per_value,dtype
meal,2,24.5,object
booking status,2,24.5,object
car_parking,2,24.5,bool
room type,3,16.3,object
source,4,12.2,object
lead_time,38,1.3,int64
price,40,1.2,float64


The top four are the transitive dependencies: 2, 3, 4 and 2 distinct values spread over 49
rows, each a category name stored once per booking. `lead_time` and `price` have high
cardinality because they are genuine measures of the booking itself — they stay.

Each category becomes a lookup table. Besides removing repetition this makes the valid set
**enforceable**: the 0NF table would accept any spelling of `Room_Type 1`.

In [8]:
LOOKUPS = {
    "meal":           ("dim_meal_plan",      "meal_plan_id"),
    "room type":      ("dim_room_type",      "room_type_id"),
    "source":         ("dim_booking_source", "source_id"),
    "booking status": ("dim_booking_status", "status_id"),
}

dims, fact = {}, one_nf.copy()
for col, (table, key) in LOOKUPS.items():
    values = sorted(fact[col].unique())
    dims[table] = pd.DataFrame({key: range(1, len(values) + 1), "nama": values})
    fact[key] = fact[col].map(dict(zip(dims[table]["nama"], dims[table][key])))
    fact = fact.drop(columns=[col])

for table, d in dims.items():
    print(f"{table:<22} {len(d)} rows  ->  {list(d['nama'])}")

dim_meal_plan          2 rows  ->  ['Meal Plan 1', 'Meal Plan 2']
dim_room_type          3 rows  ->  ['Room_Type 1', 'Room_Type 2', 'Room_Type 4']
dim_booking_source     4 rows  ->  ['Aviation', 'Corporate', 'Offline', 'Online']
dim_booking_status     2 rows  ->  ['Canceled', 'Not_Canceled']


In [9]:
# how much repetition the decomposition removes
before = sum(one_nf[c].map(lambda v: len(str(v))).sum() for c in LOOKUPS)
after  = sum(dims[t]["nama"].map(len).sum() for t, _ in LOOKUPS.values())
print(f"category text stored: {before} chars in 0NF -> {after} chars in 3NF "
      f"({100 * (1 - after / before):.0f}% less)")

category text stored: 1920 chars in 0NF -> 105 chars in 3NF (95% less)


## Step 4 — Create the schema in PostgreSQL

This is the step pandas cannot do. A normal form is a property of a **schema**, and so far
all that exists is a set of dataframes: nothing prevents a duplicate key, an orphaned
foreign key, or a nonsense value.

The DDL lives in `sql/01_schema.sql` so there is a single source of truth, and is executed
here. The constraints it declares — `PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE`, `CHECK` — are
upheld by the database on every insert from now on, including ones made years later by code
that has never seen this notebook.

In [10]:
schema_sql = (ROOT / "sql" / "01_schema.sql").read_text(encoding="utf-8")

with engine.begin() as con:
    con.execute(text(schema_sql))

tables = sorted(inspect(engine).get_table_names())
print(f"{len(tables)} tables created: {tables}")

6 tables created: ['booking', 'dim_booking_source', 'dim_booking_status', 'dim_meal_plan', 'dim_room_type', 'guest']


In [11]:
# the constraints that now enforce the design
q = """
SELECT tc.table_name, tc.constraint_type, COUNT(*) AS n
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'public'
  AND tc.constraint_type IN ('PRIMARY KEY','FOREIGN KEY','UNIQUE','CHECK')
GROUP BY tc.table_name, tc.constraint_type
ORDER BY tc.table_name, tc.constraint_type
"""
pd.read_sql(q, engine)

,table_name,constraint_type,n
0,booking,CHECK,23
1,booking,FOREIGN KEY,5
2,booking,PRIMARY KEY,1
3,dim_booking_source,CHECK,2
4,dim_booking_source,PRIMARY KEY,1
5,dim_booking_source,UNIQUE,1
6,dim_booking_status,CHECK,2
7,dim_booking_status,PRIMARY KEY,1
8,dim_booking_status,UNIQUE,1
9,dim_meal_plan,CHECK,2


## Step 5 — Load

Dimensions first, then bookings — the foreign keys require the parents to exist, which is
the point. `pandas.to_sql` appends to the tables the DDL already defined, so the declared
types and constraints are the ones in force rather than types pandas guesses.

In [12]:
with engine.begin() as con:
    for table, d in dims.items():
        d.to_sql(table, con, if_exists="append", index=False)

    fact[["guest_id"]].drop_duplicates().to_sql("guest", con,
                                                if_exists="append", index=False)

    booking_cols = ["booking_id", "guest_id", "meal_plan_id", "room_type_id",
                    "source_id", "status_id", "reservation_date", "adults",
                    "children", "weekend_nights", "week_nights", "lead_time",
                    "price", "special_requests", "car_parking"]
    fact[booking_cols].to_sql("booking", con, if_exists="append", index=False)

pd.read_sql("""
SELECT relname AS table, n_live_tup AS rows
FROM pg_stat_user_tables ORDER BY relname
""", engine)

,table,rows
0,booking,0
1,dim_booking_source,0
2,dim_booking_status,0
3,dim_meal_plan,0
4,dim_room_type,0
5,guest,0


### The constraints are real

Worth demonstrating rather than claiming. Two inserts the 0NF spreadsheet would have
accepted without complaint:

In [13]:
from sqlalchemy.exc import IntegrityError

# 1. a booking referencing a room type that does not exist
try:
    with engine.begin() as con:
        con.execute(text("""
            INSERT INTO booking (booking_id, guest_id, meal_plan_id, room_type_id,
                                 source_id, status_id, reservation_date, adults,
                                 children, weekend_nights, week_nights, lead_time,
                                 price, special_requests, car_parking)
            VALUES ('INN99999','GI00001',1,999,1,1,'2024-01-01',1,0,1,1,10,50.0,0,false)
        """))
    print("1. orphan foreign key: ACCEPTED  <- bad")
except IntegrityError as e:
    print(f"1. orphan foreign key rejected: {str(e.orig).strip().splitlines()[0]}")

# 2. a booking with no nights at all
try:
    with engine.begin() as con:
        con.execute(text("""
            INSERT INTO booking (booking_id, guest_id, meal_plan_id, room_type_id,
                                 source_id, status_id, reservation_date, adults,
                                 children, weekend_nights, week_nights, lead_time,
                                 price, special_requests, car_parking)
            VALUES ('INN99998','GI00001',1,1,1,1,'2024-01-01',1,0,0,0,10,50.0,0,false)
        """))
    print("2. zero-night booking: ACCEPTED  <- bad")
except IntegrityError as e:
    print(f"2. zero-night booking rejected: {str(e.orig).strip().splitlines()[0]}")

1. orphan foreign key rejected: insert or update on table "booking" violates foreign key constraint "booking_room_type_id_fkey"
2. zero-night booking rejected: new row for relation "booking" violates check constraint "booking_check"


## Step 6 — Is the decomposition lossless?

A decomposition that cannot be rejoined has destroyed information, and column-slicing can
do that silently. The test: rejoin everything in SQL and compare the result to the 1NF
table, column for column.

In [14]:
views_sql = (ROOT / "sql" / "02_views.sql").read_text(encoding="utf-8")
with engine.begin() as con:
    con.execute(text(views_sql))

rebuilt = pd.read_sql("SELECT * FROM v_booking_flat ORDER BY booking_id", engine)
print(f"rebuilt {len(rebuilt)} rows from the normalised tables")
rebuilt.head(3)

rebuilt 49 rows from the normalised tables


,booking_id,guest_id,reservation_date,adults,children,weekend_nights,week_nights,total_nights,meal_plan,room_type,booking_source,booking_status,lead_time,price,special_requests,car_parking
0,INN00001,GI00001,2015-02-10,1,1,2,5,7,Meal Plan 1,Room_Type 1,Offline,Not_Canceled,224,88.00,0,False
1,INN00002,GI00002,2018-06-11,1,0,1,3,4,Meal Plan 1,Room_Type 1,Online,Not_Canceled,5,106.68,1,False
2,INN00003,GI00003,2018-02-28,2,1,1,3,4,Meal Plan 1,Room_Type 2,Online,Canceled,1,50.00,0,False


In [15]:
shared = ["booking_id", "guest_id", "reservation_date", "adults", "children",
          "weekend_nights", "week_nights", "lead_time", "price",
          "special_requests", "car_parking"]
renamed = {"meal": "meal_plan", "room type": "room_type",
           "source": "booking_source", "booking status": "booking_status"}

a = one_nf.rename(columns=renamed).sort_values("booking_id").reset_index(drop=True)
b = rebuilt.sort_values("booking_id").reset_index(drop=True)

checks = []
for c in shared + list(renamed.values()):
    left, right = a[c], b[c]
    if pd.api.types.is_float_dtype(left):
        same = bool((left.round(2) == right.astype(float).round(2)).all())
    else:
        same = bool((left.astype(str).values == right.astype(str).values).all())
    checks.append({"column": c, "identical": same})

result = pd.DataFrame(checks)
print(f"rows: source {len(a)}  rebuilt {len(b)}")
print(f"columns identical: {result.identical.sum()} / {len(result)}")
assert len(a) == len(b) and result.identical.all(), "decomposition is NOT lossless"
print("\nlossless: the 3NF tables reproduce the 1NF table exactly")
result.T

rows: source 49  rebuilt 49
columns identical: 15 / 15

lossless: the 3NF tables reproduce the 1NF table exactly


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
column,booking_id,guest_id,reservation_date,adults,children,weekend_nights,week_nights,lead_time,price,special_requests,car_parking,meal_plan,room_type,booking_source,booking_status
identical,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


In [16]:
# totals as a second, independent check
pd.read_sql("""
SELECT COUNT(*)                                AS bookings,
       ROUND(SUM(price)::numeric, 2)           AS price_total,
       SUM(weekend_nights + week_nights)       AS nights_total
FROM booking
""", engine).assign(
    source_price=round(float(one_nf.price.sum()), 2),
    source_nights=int((one_nf.weekend_nights + one_nf.week_nights).sum()),
)

,bookings,price_total,nights_total,source_price,source_nights
0,49,4489.93,141,4489.93,141


## Step 7 — What the normalised schema makes easy

Each of these needs a join across the lookup tables, which is what normalising bought: the
category names exist in exactly one place, so renaming `Not_Canceled` is one `UPDATE`
rather than 49.

In [17]:
pd.read_sql("SELECT * FROM v_cancellation_by_source", engine)

,booking_source,bookings,cancelled,cancel_pct,avg_lead_time
0,Online,32,13,40.6,72.0
1,Offline,15,1,6.7,100.0
2,Aviation,1,1,100.0,2.0
3,Corporate,1,0,0.0,122.0


In [18]:
pd.read_sql("SELECT * FROM v_revenue_by_room_type", engine)

,room_type,bookings,avg_price,avg_nights,est_revenue
0,Room_Type 1,31,88.38,2.9,8044.44
1,Room_Type 4,9,104.79,3.2,3073.84
2,Room_Type 2,9,89.67,2.3,1715.52


In [19]:
pd.read_sql("SELECT * FROM v_lead_time_vs_cancellation", engine)

,lead_bucket,bookings,cancelled,cancel_pct
0,0-6 days,9,2,22.2
1,7-29 days,2,0,0.0
2,30-89 days,21,5,23.8
3,90+ days,17,8,47.1


Cancellation appears to rise with lead time, and online bookings appear to cancel far more
than offline. Both are tested in `significance_check.py`, and the answers are not what the
tables suggest: with 49 bookings the lead-time effect is **borderline** (p = 0.045 by one
test, 0.106 by another) and the four-way source comparison is **invalid** (5 of 8 expected
cell counts below 5). Only Online vs Offline survives — Fisher exact, odds ratio 9.58,
p = 0.0199.

A ranked table always produces a winner, which is why the test belongs before the
narrative.

---

## Summary

| | |
|---|---|
| 0NF source | 49 rows × 15 columns, no keys, no constraints |
| 1NF | mixed date formats parsed, `parking` text → boolean, names normalised |
| 2NF | already satisfied — no composite key exists, so no partial dependency |
| 3NF | 4 category attributes extracted to lookup tables, 95% less repeated text |
| Enforcement | PostgreSQL keys, foreign keys and checks, demonstrated by two rejected inserts |
| Lossless | verified — the normalised tables reproduce the 1NF table column for column |